# Bab 5 - Latihan Bonus: Membandingkan Strategi Memori

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Nur-Azzam-Wafiuddin/workshop/blob/uji-colab/colab/hari3/05-memori-agen-latihan-2.ipynb)

Ini versi **Google Colab**. Isi dan urutan contohnya sama dengan versi VS Code; yang berbeda hanya cell Persiapan di bawah, yang menyalin repository, memasang pustaka, dan mengisi API key di runtime Colab.

1. Pilih **File > Save a copy in Drive** agar perubahan tersimpan di akun Anda sendiri.
2. Jalankan tiga cell **Persiapan Colab** berurutan (sekitar 2-3 menit pada sesi baru).
3. Kerjakan dari atas ke bawah dengan `Shift+Enter`.

Kode jawaban latihan ada di bagian **Kode Jawaban Latihan** pada [PDF bab ini](https://github.com/Nur-Azzam-Wafiuddin/workshop/blob/uji-colab/hari3/05-memori-agen/05-memori-agen.pdf). Salin kodenya ke cell yang bertanda `TODO`.

Kalau runtime terputus, file di `/content` (termasuk `.env`) hilang. Jalankan ulang Persiapan, lalu lanjutkan dari cell yang sedang Anda kerjakan.

## Persiapan Colab
Jalankan tiga cell berikut berurutan sebelum cell lain di notebook ini.

In [ ]:
# Persiapan 1 - Menyalin repository dan memasang pustaka (sekitar 2-3 menit pada sesi baru)
import os
import subprocess
import sys

REPO_URL = "https://github.com/Nur-Azzam-Wafiuddin/workshop.git"
REPO = "/content/workshop-genai-migas" if os.path.isdir("/content") else os.path.abspath("workshop-genai-migas")
if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "-q", "-b", "uji-colab", REPO_URL, REPO], check=True)
os.chdir(REPO)

pasang = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "colab/requirements-colab.txt", "-e", "."],
                        capture_output=True, text=True)
if pasang.returncode != 0:
    print(pasang.stderr[-3000:])
    raise RuntimeError("Instalasi pustaka gagal. Jalankan ulang cell ini.")

# "-e ." memasang common dan migas_shared agar subproses (server MCP) dapat mengimpornya; kernel yang sedang berjalan
# belum memuat instalasi baru, jadi folder repo juga ditambahkan ke sys.path
if REPO not in sys.path:
    sys.path.insert(0, REPO)
os.environ["PYTHONPATH"] = REPO + os.pathsep + os.environ.get("PYTHONPATH", "")
print("Repository dan pustaka siap. Folder kerja:", os.getcwd())

In [ ]:
# Persiapan 2 - Mengisi ID peserta dan API key (dilewati bila .env sudah berisi key)
import re
from getpass import getpass
from pathlib import Path

from dotenv import load_dotenv

env = Path(REPO) / ".env"


def rahasia(nama: str):
    """Nilai dari Colab Secrets (ikon kunci di panel kiri), atau None bila tidak ada."""
    try:
        from google.colab import userdata
        return userdata.get(nama)
    except Exception:  # noqa: BLE001 - bukan di Colab, secret belum dibuat, atau akses notebook belum diizinkan
        return None


if not (env.exists() and re.search(r"(?m)^ONLINE_API_KEY=(?!isi-dengan)\S+", env.read_text(encoding="utf-8"))):
    peserta = (rahasia("PARTICIPANT_ID") or input("ID peserta (nomor meja dan nama singkat, contoh P07-andi): ")).strip()
    kunci = (rahasia("ONLINE_API_KEY") or getpass("API key workshop (ketikan tidak ditampilkan): ")).strip()
    if not kunci:
        raise ValueError("API key belum diisi. Jalankan ulang cell ini.")
    isi = (Path(REPO) / ".env.example").read_text(encoding="utf-8")
    for nama, nilai in (("PARTICIPANT_ID", peserta or "P00-nama"), ("ONLINE_API_KEY", kunci)):
        isi = re.sub(rf"(?m)^{nama}=.*$", lambda _, n=nama, v=nilai: f"{n}={v}", isi)
    env.write_text(isi, encoding="utf-8")
    del kunci

# Nilai dari .env dimuat ulang; modul yang sudah diimpor dibuang agar membaca nilai terbaru
load_dotenv(env, override=True)
for modul in [m for m in sys.modules if m.split(".")[0] in ("common", "migas_shared", "rag_pipeline")]:
    del sys.modules[modul]
print("File .env siap untuk peserta", os.environ.get("PARTICIPANT_ID"))

In [ ]:
# Persiapan 3 - Memeriksa koneksi ke model
from migas_shared.llm import chat

try:
    print("Jawaban model:", chat("Balas hanya dengan kata: SIAP").strip())
except Exception as error:  # noqa: BLE001
    raise RuntimeError("Model tidak menjawab. Periksa API key (jalankan ulang Persiapan 2) dan koneksi internet. "
                       f"Pesan asli: {error}") from error

Isi bagian bertanda `TODO(...)` dengan kode dari bagian **Kode Jawaban Latihan** pada PDF bab ini, lalu jalankan sel satu per satu
dengan Shift+Enter. Baris `... = None  # TODO` atau `raise NotImplementedError` berarti kode itu harus Anda ganti.

Menjalankan percakapan berskrip YANG SAMA dengan tiga strategi dan mengukur:
- berapa banyak pesan dan (perkiraan) token yang tersimpan di memori
- apakah agen masih bisa mengingat fakta awal di akhir percakapan

Strategi: simpan semua, pangkas ke N pesan terakhir, ringkas pesan lama.

In [ ]:
# Langkah 1 - Menyiapkan model
import os

from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware, before_model
from langchain_core.messages import HumanMessage, RemoveMessage
from langchain_core.messages.utils import count_tokens_approximately
from langchain_openai import ChatOpenAI
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph.message import REMOVE_ALL_MESSAGES

load_dotenv()

print("Membandingkan Strategi Memori\n")

model = ChatOpenAI(
    model=os.getenv("ONLINE_MODEL"),
    base_url=os.getenv("ONLINE_BASE_URL"),
    api_key=os.getenv("ONLINE_API_KEY"),
    reasoning_effort=os.getenv("LLM_REASONING_EFFORT", "none"),
)
prompt = "Anda asisten operasi yang membantu. Jawab singkat dalam Bahasa Indonesia."

## Langkah 2 - Percakapan uji dan fungsi evaluasi
`evaluasi` memutar percakapan berskrip, mengajukan pertanyaan pengingat di akhir, lalu mengukur isi memori.

In [ ]:
SIMPAN_TERAKHIR = 4

PERCAKAPAN = [
    "Nomor Permit to Work yang saya pegang adalah PTW-2026-0714.",
    "Pekerjaan kami adalah penggantian gasket flange pada kompresor K-ML1-201.",
    "Pekan lalu terjadi pemadaman listrik selama 2 jam di GS-ML1.",
    "Generator cadangan menyala otomatis setelah 30 detik.",
    "Kemarin kompresor menangani sekitar 1,2 juta scf gas.",
]
PERTANYAAN_INGAT = "Apa nomor Permit to Work yang saya sebutkan di awal?"
DIHARAPKAN = "PTW-2026-0714"


# TODO(B5.4): buat middleware @before_model simpan_terbaru yang menyisakan SIMPAN_TERAKHIR pesan terakhir (hapus sisanya dengan RemoveMessage(id=REMOVE_ALL_MESSAGES))
simpan_terbaru = None  # TODO: ganti dengan implementasi Anda


def evaluasi(nama: str, agent) -> dict:
    config = {"configurable": {"thread_id": nama}}
    for teks in PERCAKAPAN:
        agent.invoke({"messages": [HumanMessage(content=teks)]}, config)

    jawaban = agent.invoke({"messages": [HumanMessage(content=PERTANYAAN_INGAT)]}, config)
    tersimpan = agent.get_state(config).values["messages"]
    return {
        "strategi": nama,
        "pesan": len(tersimpan),
        "token": count_tokens_approximately(tersimpan),
        "ingat": DIHARAPKAN in jawaban["messages"][-1].content,
        "jawaban": jawaban["messages"][-1].content,
    }

## Langkah 3 - Tiga agen dengan strategi berbeda

In [ ]:
# TODO(B5.5): buat tiga agen tanpa tool, masing-masing dengan InMemorySaver: 'simpan semua' (tanpa middleware), 'pangkas ke N' (middleware simpan_terbaru), dan 'ringkas' (SummarizationMiddleware dengan trigger ('messages', 6) dan keep ('messages', 2))
agen = {}  # TODO: ganti dengan implementasi Anda

hasil = []

## Langkah 4 - Strategi 1: simpan semua

In [ ]:
nama = "simpan semua"
print(f"Menjalankan: {nama}")
hasil.append(evaluasi(nama, agen[nama]))

## Langkah 5 - Strategi 2: pangkas ke pesan terakhir

In [ ]:
nama = f"pangkas ke {SIMPAN_TERAKHIR}"
print(f"Menjalankan: {nama}")
hasil.append(evaluasi(nama, agen[nama]))

## Langkah 6 - Strategi 3: ringkas pesan lama

In [ ]:
nama = "ringkas"
print(f"Menjalankan: {nama}")
hasil.append(evaluasi(nama, agen[nama]))

## Langkah 7 - Tabel perbandingan

In [ ]:
print(f"{'Strategi':<20}{'Pesan':>8}{'~Token':>10}   Ingat nomor PTW?")
print("-" * 60)
for h in hasil:
    tanda = "ya" if h["ingat"] else "tidak"
    print(f"{h['strategi']:<20}{h['pesan']:>8}{h['token']:>10}   {tanda}")

## Membaca tabel
- Simpan semua: mengingat, tetapi tumbuh tanpa batas
- Pangkas: kecil, tetapi bisa melupakan fakta awal
- Ringkas: menjaga fakta kunci, tetapi ringkasannya sendiri memakai token.
  Pada percakapan PENDEK ringkasan bisa lebih besar dari aslinya; manfaatnya terasa
  pada percakapan panjang (coba perpanjang daftar `PERCAKAPAN`)